# ABR Geocoder（デジタル庁公式）検証用Notebook

このNotebookは、**デジタル庁公式 [`digital-go-jp/abr-geocoder`](https://github.com/digital-go-jp/abr-geocoder) が
Google Colab上で現在成立するかどうかを確認するための検証用Notebook**です。

- **sheltermatch本体（`sheltermatch.ipynb`）ではありません。**
- **`experiments/jageocoder_poc.ipynb`ではありません。**
- **Jageocoderの代替採用を決定するNotebookでもありません。**
- 今回確認するのは「現在の公式構成（`abrdb` + `abrg` + PostgreSQL + DuckDB）がColab上で成立するか」だけです。

## 背景

`experiments/jageocoder_poc.ipynb`では、Jageocoder 2.2.1.1・jageocoder-converter v2.1.1とも
importまでは成功しましたが、辞書生成時にアドレス・ベース・レジストリ（ABR）の旧配布先
`catalog.registries.digital.go.jp`を参照しており、現在のColabから名前解決できず停止しました。

今回、公式`abr-geocoder`のソースコードを確認したところ、現在の公式配布先は
`https://dataset.address-br.digital.go.jp/api/feed/dcat-us/1.1.json`へ変更されていることを確認しました
（`abrdb/internal/infra/config/config.go`の`DefaultFeedURL`）。したがって、jageocoder-converter側の
古い参照先を独自修正するのではなく、**現在メンテナンスされている公式`abr-geocoder`そのもの**を検証します。

## 公式構成（README確認済み）

- `abrdb`: ABRデータをPostgreSQLへ取り込むCLI
- `abrg`: PostgreSQLからDuckDBキャッシュを構築し、CLI/APIで住所マッチング・ジオコーディングを行う

今回は沖縄県（`--pref 47`）・全カテゴリ（`--category all`）・座標付き（`--pos`）に限定します。
**全国データは取得しません。** APIサーバーの常駐化も行わず、公式`abrg geocode` CLIで検証します。

## 進め方

上から順にすべてのセルを実行してください（「ランタイム → すべてのセルを実行」）。

外部データ・外部リソースの取得に失敗した場合、このNotebookは失敗を握りつぶさず、失敗した段階・取得先・
HTTPステータス/DNSエラー等・エラー概要を表示して停止します。以下は行いません。

- Jageocoderへの自動フォールバック
- 非公式ABRミラー・旧`catalog.registries.digital.go.jp`の利用
- Google Maps等の外部Geocoding APIの利用
- 独自スクレイピング、公式`abr-geocoder`ソースの仕様改変
- 全国データへの切替
- Dockerが不要であることを確認しないままのDocker導入（まずネイティブビルドを優先します）
- 失敗を握りつぶして次工程を成功扱いにすること


In [ ]:
# ===== 設定 =====

PREF_CODE = "47"
TARGET_CITY = "糸満市"
ABR_CATEGORY = "all"
ENABLE_POSITION = True

# 以下は通常変更不要な内部設定。
WORK_DIR = "./abr_geocoder_poc_work"
REPO_URL = "https://github.com/digital-go-jp/abr-geocoder.git"
REPO_DIR = f"{WORK_DIR}/abr-geocoder"
GO_INSTALL_DIR = f"{WORK_DIR}/go-toolchain"

# PoC用のローカルPostgreSQL設定（このNotebook内でのみ使用し、外部公開しない簡易パスワード）。
DB_HOST = "127.0.0.1"
DB_PORT = "5432"
DB_USER = "postgres"
DB_PASSWORD = "abr-geocoder-poc-local-only"
DB_NAME = "abrdb_poc"
DB_SSLMODE = "disable"
DB_ENV = {
    "DB_HOST": DB_HOST,
    "DB_PORT": DB_PORT,
    "DB_USER": DB_USER,
    "DB_PASSWORD": DB_PASSWORD,
    "DB_NAME": DB_NAME,
    "DB_SSLMODE": DB_SSLMODE,
}

# 個人情報は使用しない。糸満市内の公開されている公共施設の住所
# （experiments/jageocoder_poc.ipynbと同じもの）を流用する。
TEST_ADDRESSES = [
    "沖縄県糸満市潮崎町1丁目1番地",  # 糸満市役所
    "沖縄県糸満市字糸満673",  # 糸満市立糸満小学校
    "沖縄県糸満市真栄里1448番地",  # 糸満市立中央図書館
]

print("設定を読み込みました。")
print(f"  PREF_CODE      = '{PREF_CODE}' ({TARGET_CITY}を含む沖縄県)")
print(f"  ABR_CATEGORY   = '{ABR_CATEGORY}'")
print(f"  ENABLE_POSITION = {ENABLE_POSITION}")


In [ ]:
# ===== Colab環境確認 =====
import io
import json
import os
import platform
import re
import shutil
import subprocess
import sys

import pandas as pd
from google.colab import files


def run_cmd(cmd, env=None, cwd=None, timeout=None):
    """外部コマンドを実行しCompletedProcessを返す（例外は呼び出し側で判断する）。"""
    merged_env = {**os.environ, **(env or {})}
    try:
        return subprocess.run(
            cmd, cwd=cwd, env=merged_env, capture_output=True, text=True, timeout=timeout
        )
    except (FileNotFoundError, subprocess.TimeoutExpired) as e:
        return subprocess.CompletedProcess(cmd, returncode=1, stdout="", stderr=f"{type(e).__name__}: {e}")


def _first_line(text, default):
    text = (text or "").strip()
    return text.splitlines()[0] if text else default


print(f"OS            : {platform.platform()}")
print(f"Python version: {sys.version.splitlines()[0]}")

go_check = run_cmd(["go", "version"])
print(f"Go version    : {_first_line(go_check.stdout + go_check.stderr, '未導入')}")

psql_check = run_cmd(["psql", "--version"])
print(f"PostgreSQLクライアント: {_first_line(psql_check.stdout + psql_check.stderr, '未導入')}")

pg_server_check = run_cmd(["pg_lsclusters"])
print("PostgreSQLサーバー:")
print("  " + (pg_server_check.stdout.strip() or "見つかりません（次のセルで導入します）").replace("\n", "\n  "))

disk = shutil.disk_usage(".")
print(f"空きディスク容量: {disk.free / (1024 ** 3):.1f} GB（全体 {disk.total / (1024 ** 3):.1f} GB）")

MEM_TOTAL_GB = None
try:
    with open("/proc/meminfo") as f:
        for line in f:
            if line.startswith("MemTotal:"):
                MEM_TOTAL_GB = int(line.split()[1]) / (1024 ** 2)
                break
except FileNotFoundError:
    pass
print(f"メモリ量: {MEM_TOTAL_GB:.1f} GB" if MEM_TOTAL_GB else "メモリ量: 取得できませんでした")

print()
print(
    "このPoCは沖縄県全カテゴリ・座標付きデータとPostgreSQL/DuckDBを扱います。"
    "必要最低量を独自に決め打ちすることはしないため、上記の実測値を踏まえて後続処理を進めてください。"
)

poc_status = {}


In [ ]:
# ===== 公式リポジトリ取得・バージョン確認 =====
# 公式 digital-go-jp/abr-geocoder のみを利用する（非公式forkは使用しない）。

os.makedirs(WORK_DIR, exist_ok=True)

if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    print(f"'{REPO_DIR}' は取得済みのため、再取得しません。")
    repo_fetch_ok = True
else:
    print(f"公式リポジトリ（{REPO_URL}）のmainブランチのみを取得します。")
    clone = run_cmd(["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, REPO_DIR], timeout=300)
    repo_fetch_ok = clone.returncode == 0
    if not repo_fetch_ok:
        print("公式リポジトリの取得に失敗しました。")
        print((clone.stdout + clone.stderr)[-2000:])

poc_status["repo_fetch"] = repo_fetch_ok

if repo_fetch_ok:
    commit_sha = run_cmd(["git", "-C", REPO_DIR, "rev-parse", "HEAD"]).stdout.strip()
    commit_date = run_cmd(["git", "-C", REPO_DIR, "log", "-1", "--format=%ci"]).stdout.strip()
    print(f"commit SHA : {commit_sha}")
    print(f"commit date: {commit_date}")

    def _read_version_file(path):
        try:
            with open(path) as f:
                return f.read().strip()
        except FileNotFoundError:
            return "(VERSIONファイルなし)"

    print(
        "abrdb/VERSION (ビルド前の埋め込み予定バージョン):",
        _read_version_file(os.path.join(REPO_DIR, "abrdb", "VERSION")),
    )
    print(
        "abrg/VERSION  (ビルド前の埋め込み予定バージョン):",
        _read_version_file(os.path.join(REPO_DIR, "abrg", "VERSION")),
    )
    print("実際にビルドしたバイナリのバージョンは次のセルで `abrdb version` / `abrg version` により確認する。")


In [ ]:
# ===== abrdb / abrg のビルド（ネイティブ、Dockerは使用しない） =====

ABRDB_BIN = os.path.abspath(os.path.join(REPO_DIR, "abrdb", "abrdb"))
ABRG_BIN = os.path.abspath(os.path.join(REPO_DIR, "abrg", "abrg"))

DOCKER_REQUIRED_HINTS = ("cannot open shared object file", 'exec: "cc"', "cgo: C compiler")


def _ensure_go():
    check = run_cmd(["go", "version"])
    if check.returncode == 0:
        print(f"Go: {check.stdout.strip()}（既存の導入を利用）")
        return True

    print("Goが見つからないため、公式配布元(go.dev)から導入する。")
    version_result = run_cmd(["curl", "-fsSL", "https://go.dev/VERSION?m=text"], timeout=60)
    if version_result.returncode != 0 or not version_result.stdout.strip():
        print("Goの最新バージョン情報を取得できなかった。")
        print((version_result.stdout + version_result.stderr)[-1000:])
        return False

    go_version = version_result.stdout.strip().splitlines()[0]
    tarball_url = f"https://go.dev/dl/{go_version}.linux-amd64.tar.gz"
    os.makedirs(GO_INSTALL_DIR, exist_ok=True)
    tarball_path = os.path.join(GO_INSTALL_DIR, "go.tar.gz")
    print(f"{tarball_url} から導入する。")
    download = run_cmd(["curl", "-fsSL", "-o", tarball_path, tarball_url], timeout=300)
    if download.returncode != 0:
        print("Goのダウンロードに失敗した。")
        print((download.stdout + download.stderr)[-1000:])
        return False

    extract = run_cmd(["tar", "-C", GO_INSTALL_DIR, "-xzf", tarball_path])
    if extract.returncode != 0:
        print("Goの展開に失敗した。")
        print((extract.stdout + extract.stderr)[-1000:])
        return False

    go_bin = os.path.abspath(os.path.join(GO_INSTALL_DIR, "go", "bin"))
    os.environ["PATH"] = go_bin + os.pathsep + os.environ.get("PATH", "")
    verify = run_cmd(["go", "version"])
    print(f"導入したGo: {verify.stdout.strip()}")
    return verify.returncode == 0


if not poc_status.get("repo_fetch"):
    print("公式リポジトリの取得に失敗しているため、このセルはスキップする。")
    poc_status["abrdb_build"] = False
    poc_status["abrg_build"] = False
elif not _ensure_go():
    print("Goの導入に失敗したため、ビルドをスキップする。")
    poc_status["abrdb_build"] = False
    poc_status["abrg_build"] = False
else:
    for module_name, bin_path in (("abrdb", ABRDB_BIN), ("abrg", ABRG_BIN)):
        module_dir = os.path.join(REPO_DIR, module_name)
        print(f"\n{module_name} をビルドする（go build。Dockerは使用しない）。")
        build = run_cmd(["make", "build"], cwd=module_dir, timeout=900)
        ok = build.returncode == 0 and os.path.exists(bin_path)
        poc_status[f"{module_name}_build"] = ok
        build_output = build.stdout + build.stderr
        if not ok:
            print(f"{module_name} のビルドに失敗した。")
            print(build_output[-3000:])
            if any(hint in build_output for hint in DOCKER_REQUIRED_HINTS):
                print(
                    f"{module_name} はこの環境ではネイティブビルドできない可能性がある"
                    "（cコンパイラ・共有ライブラリ関連のエラー）。"
                    "現時点の公式README・Makefile構成ではDockerは必須と明記されていないため、"
                    "Docker環境は無理に構築せず、ここで停止する。"
                )
        else:
            version = run_cmd([bin_path, "version"])
            print(f"{module_name} version: {version.stdout.strip()}")


In [ ]:
# ===== PostgreSQL準備（ローカル・PoC専用） =====
# DB_HOST/DB_PORT/DB_USER/DB_PASSWORD/DB_NAME/DB_SSLMODEは設定セルのDB_ENVを使う。
# PoC用のローカルDBであり、簡易パスワードのまま外部公開はしない。

pg_check = run_cmd(["pg_lsclusters"])
if not pg_check.stdout.strip():
    print("PostgreSQLサーバーが見つからないため、導入する。")
    install = run_cmd(["bash", "-c", "apt-get update -qq && apt-get install -y -qq postgresql"], timeout=300)
    if install.returncode != 0:
        print("PostgreSQLの導入に失敗した。")
        print((install.stdout + install.stderr)[-2000:])
        poc_status["postgres"] = False

if poc_status.get("postgres") is not False:
    start = run_cmd(["service", "postgresql", "start"])
    print((start.stdout + start.stderr).strip() or "PostgreSQLサービスを起動した。")

    run_cmd(["sudo", "-u", "postgres", "psql", "-c", f"ALTER USER postgres PASSWORD '{DB_PASSWORD}';"])
    create_db = run_cmd(["sudo", "-u", "postgres", "psql", "-c", f"CREATE DATABASE {DB_NAME};"])
    if create_db.returncode != 0 and "already exists" not in (create_db.stdout + create_db.stderr):
        print("データベース作成でエラーが発生した。")
        print((create_db.stdout + create_db.stderr)[-1000:])

    verify = run_cmd(
        ["psql", "-h", DB_HOST, "-p", DB_PORT, "-U", DB_USER, "-d", DB_NAME, "-c", "SELECT version();"],
        env={"PGPASSWORD": DB_PASSWORD},
    )
    postgres_ok = verify.returncode == 0
    poc_status["postgres"] = postgres_ok
    if postgres_ok:
        print(f"接続成功: DB '{DB_NAME}' へ {DB_HOST}:{DB_PORT} 経由で接続できた。")
        print(f"DB作成成功: '{DB_NAME}'")
    else:
        print("PostgreSQLへの接続に失敗した。")
        print((verify.stdout + verify.stderr)[-2000:])


In [ ]:
# ===== abrdb初期化（沖縄県限定） =====

if not (poc_status.get("abrdb_build") and poc_status.get("postgres")):
    print("abrdbのビルドまたはPostgreSQLの準備が完了していないため、このセルはスキップする。")
    poc_status["okinawa_init"] = False
else:
    init_args = [ABRDB_BIN, "init", "--pref", PREF_CODE, "--category", ABR_CATEGORY, "--force"]
    if ENABLE_POSITION:
        init_args.append("--pos")
    print(f"沖縄県（{PREF_CODE}）・カテゴリ（{ABR_CATEGORY}）限定で初期化する（全国データは対象にしない）。")
    init_result = run_cmd(init_args, env=DB_ENV, timeout=120)
    init_ok = init_result.returncode == 0
    poc_status["okinawa_init"] = init_ok
    print((init_result.stdout + init_result.stderr).strip())

    if init_ok:
        config = run_cmd([ABRDB_BIN, "show", "config"], env=DB_ENV)
        # 埋め込みimportプロファイルの全文(YAML)は表示せず、概要部分だけを表示する。
        summary = config.stdout.split("Import config")[0].strip()
        print(summary)
    else:
        print("abrdb initに失敗した。")
        print((init_result.stdout + init_result.stderr)[-2000:])


In [ ]:
# ===== ABRデータ取得・PostgreSQLへのimport =====


def _extract_failure_info(text):
    """失敗メッセージからURLとHTTP status/DNSエラー等の手がかりを抽出する
    （見つからない場合は推測せずNoneを返す）。"""
    url_match = re.search(r'https?://[^\s")]+', text)
    status_match = re.search(r"\b([1-5]\d{2})\b", text)
    lower = text.lower()
    if status_match:
        status_text = status_match.group(1)
    elif "forbidden" in lower:
        status_text = "Forbidden"
    elif "no such host" in lower or "could not resolve" in lower or "dns" in lower:
        status_text = "DNS解決エラー"
    elif "timeout" in lower or "timed out" in lower:
        status_text = "タイムアウト"
    else:
        status_text = None
    return (url_match.group(0) if url_match else None), status_text


if not poc_status.get("okinawa_init"):
    print("abrdb initが完了していないため、このセルはスキップする。")
    poc_status["abr_fetch"] = False
    poc_status["postgres_import"] = False
else:
    print(f"沖縄県（{PREF_CODE}）限定でABRデータを取得・importする（全国データは取得しない）。")
    import_result = run_cmd([ABRDB_BIN, "import", "--force"], env=DB_ENV, timeout=3600)
    import_ok = import_result.returncode == 0
    output_text = import_result.stdout + import_result.stderr

    # abrdb importは「ABRデータ取得」と「PostgreSQLへの反映」を1コマンドで行う設計のため、
    # 現在の公式CLI構成ではこの2項目を分離して判定できず、同じ結果をそのまま反映する。
    poc_status["abr_fetch"] = import_ok
    poc_status["postgres_import"] = import_ok

    if import_ok:
        print("ABRデータの取得・PostgreSQLへのimportに成功した。")
        digest = []
        for line in output_text.splitlines():
            try:
                entry = json.loads(line)
            except (ValueError, TypeError):
                continue
            if isinstance(entry, dict) and "msg" in entry:
                digest.append(f"[{entry.get('level', '?')}] {entry['msg']}")
        for line in digest[-5:]:
            print(f"  {line}")
    else:
        url, status_text = _extract_failure_info(output_text)
        last_line = next((l for l in reversed(output_text.strip().splitlines()) if l.strip()), "(出力なし)")
        print(f"ABR公式リポジトリ取得: {'成功' if poc_status.get('repo_fetch') else '失敗'}")
        print(f"abrdbビルド: {'成功' if poc_status.get('abrdb_build') else '失敗'}")
        print(f"PostgreSQL: {'成功' if poc_status.get('postgres') else '失敗'}")
        print(f"abrdb init: {'成功' if poc_status.get('okinawa_init') else '失敗'}")
        print("ABRデータ取得: 失敗")
        print("import: 失敗")
        print("失敗段階: abrdb import（ABRデータの取得・PostgreSQLへの反映は現在のCLIでは1コマンドのため分離不可）")
        print(f"取得先: {url or '特定できず'}")
        print(f"HTTP status / DNSエラー等: {status_text or '不明（下記エラー概要を参照）'}")
        print(f"エラー概要: {last_line}")


In [ ]:
# ===== DuckDBキャッシュ生成 =====

if not poc_status.get("postgres_import"):
    print("ABRデータのimportが完了していないため、このセルはスキップする。")
    poc_status["duckdb_cache"] = False
else:
    cache_env = dict(DB_ENV)
    if MEM_TOTAL_GB:
        # 実測メモリの約70%をABRG_CACHE_MEMORY_LIMITに設定する（固定値の決め打ちはしない）。
        cache_memory_limit_gb = max(1, int(MEM_TOTAL_GB * 0.7))
        cache_env["ABRG_CACHE_MEMORY_LIMIT"] = f"{cache_memory_limit_gb}GB"
        print(f"実測メモリ（{MEM_TOTAL_GB:.1f}GB）の約70%である{cache_memory_limit_gb}GBをABRG_CACHE_MEMORY_LIMITに設定する。")
    else:
        print("メモリ量を取得できなかったため、ABRG_CACHE_MEMORY_LIMITは公式既定値（8GB）のまま実行する。")

    # abrg cache build/geocodeは既定のキャッシュ保存先(~/.abrg/cache/)を自動作成しないため、
    # 事前にディレクトリを用意する(実機確認済み。無いと"No such file or directory"で失敗する)。
    os.makedirs(os.path.expanduser("~/.abrg/cache"), exist_ok=True)

    build = run_cmd([ABRG_BIN, "cache", "build"], env=cache_env, timeout=1800)
    cache_ok = build.returncode == 0
    poc_status["duckdb_cache"] = cache_ok
    output_text = build.stdout + build.stderr

    if not cache_ok:
        print("DuckDBキャッシュの生成に失敗した。")
        url, status_text = _extract_failure_info(output_text)
        print(f"取得先: {url or '特定できず'}")
        print(f"HTTP status / DNSエラー等: {status_text or '不明（下記エラー概要を参照）'}")
        print(f"エラー概要: {output_text.strip()[-1000:]}")
    else:
        print("DuckDBキャッシュを生成した。")
        info = run_cmd([ABRG_BIN, "cache", "info"])
        print(info.stdout.strip())


In [ ]:
# ===== 公開住所でgeocodeテスト =====
# 個人情報は使用しない。糸満市内の公開されている公共施設の住所を使う。
# 公式`abrg geocode` CLIをそのまま使用する（APIサーバーは起動しない）。

geocode_results = []

if not poc_status.get("duckdb_cache"):
    print("DuckDBキャッシュが生成されていないため、このセルはスキップする。")
    poc_status["geocode"] = False
else:
    input_path = os.path.join(WORK_DIR, "geocode_input.txt")
    output_path = os.path.join(WORK_DIR, "geocode_output.jsonl")
    with open(input_path, "w") as f:
        f.write("\n".join(TEST_ADDRESSES) + "\n")

    geocode = run_cmd(
        [ABRG_BIN, "geocode", "-i", input_path, "-o", output_path, "-c", ABR_CATEGORY, "-p", PREF_CODE, "-l", "1"],
        timeout=300,
    )
    if geocode.returncode != 0:
        print("abrg geocodeの実行に失敗した。")
        print((geocode.stdout + geocode.stderr)[-2000:])
        poc_status["geocode"] = False
    else:
        with open(output_path) as f:
            output_lines = [line for line in f if line.strip()]

        for address, line in zip(TEST_ADDRESSES, output_lines):
            entry = json.loads(line)
            if "error" in entry:
                geocode_results.append(
                    {
                        "input_address": address, "matched_address": None, "latitude": None,
                        "longitude": None, "category": None, "match_level": None,
                        "coordinates_level": None, "status": f"error: {entry['error']}",
                    }
                )
                continue

            features = entry.get("features") or []
            category = entry.get("query", {}).get("category")
            if features:
                props = features[0].get("properties", {})
                geometry = features[0].get("geometry")
                lon, lat = geometry["coordinates"] if geometry else (None, None)
                geocode_results.append(
                    {
                        "input_address": address,
                        "matched_address": props.get("matched_address"),
                        "latitude": lat,
                        "longitude": lon,
                        "category": category,
                        "match_level": props.get("match_level"),
                        "coordinates_level": props.get("coordinates_level"),
                        "status": "matched",
                    }
                )
            else:
                geocode_results.append(
                    {
                        "input_address": address, "matched_address": None, "latitude": None,
                        "longitude": None, "category": category, "match_level": None,
                        "coordinates_level": None, "status": "no_match",
                    }
                )

        matched_count = sum(1 for r in geocode_results if r["status"] == "matched")
        print(f"{len(TEST_ADDRESSES)}件中{matched_count}件を変換した。")
        poc_status["geocode"] = matched_count > 0

        display(pd.DataFrame(geocode_results))


In [ ]:
# ===== CSVアップロードによる簡易確認（検証用） =====
# 任意のセル。id,address列を持つCSVをアップロードすると一括変換を試せる。
# 住所ごとにプロセスを起動せず、公式CLIのバッチ入出力（1回のabrg geocode呼び出し）を使う。
# あくまでPoCの成立確認用であり、sheltermatch本体のCSV仕様への統合はここでは行わない。
# 変換できなかった行も削除しない。

if not poc_status.get("geocode"):
    print("公開住所でのgeocodeが成立していないため、このセルはスキップする。")
    poc_status["csv_batch"] = None
else:
    print("id,address列を持つCSVを選択してください（試さない場合はアップロードをキャンセルしてください）。")
    uploaded_csv = files.upload()

    if not uploaded_csv:
        print("CSVがアップロードされなかったため、このセルはスキップされた。")
        poc_status["csv_batch"] = None
    else:
        csv_filename = list(uploaded_csv.keys())[0]
        addresses_df = pd.read_csv(io.BytesIO(uploaded_csv[csv_filename]))

        if "address" not in addresses_df.columns:
            print("'address'列が見つからない。id,addressの形式で用意すること。")
            poc_status["csv_batch"] = False
        else:
            batch_input_path = os.path.join(WORK_DIR, "geocode_batch_input.txt")
            batch_output_path = os.path.join(WORK_DIR, "geocode_batch_output.jsonl")
            batch_addresses = addresses_df["address"].astype(str).tolist()
            with open(batch_input_path, "w") as f:
                f.write("\n".join(batch_addresses) + "\n")

            batch = run_cmd(
                [ABRG_BIN, "geocode", "-i", batch_input_path, "-o", batch_output_path,
                 "-c", ABR_CATEGORY, "-p", PREF_CODE, "-l", "1"],
                timeout=600,
            )
            if batch.returncode != 0:
                print("一括変換に失敗した。")
                print((batch.stdout + batch.stderr)[-2000:])
                poc_status["csv_batch"] = False
            else:
                with open(batch_output_path) as f:
                    output_lines = [line for line in f if line.strip()]

                matched_addresses, latitudes, longitudes, geocode_statuses = [], [], [], []
                for line in output_lines:
                    entry = json.loads(line)
                    if "error" in entry:
                        matched_addresses.append(None)
                        latitudes.append(None)
                        longitudes.append(None)
                        geocode_statuses.append(f"error: {entry['error']}")
                        continue
                    features = entry.get("features") or []
                    if features:
                        props = features[0].get("properties", {})
                        geometry = features[0].get("geometry")
                        lon, lat = geometry["coordinates"] if geometry else (None, None)
                        matched_addresses.append(props.get("matched_address"))
                        latitudes.append(lat)
                        longitudes.append(lon)
                        geocode_statuses.append("matched")
                    else:
                        matched_addresses.append(None)
                        latitudes.append(None)
                        longitudes.append(None)
                        geocode_statuses.append("no_match")

                addresses_df["matched_address"] = matched_addresses
                addresses_df["latitude"] = latitudes
                addresses_df["longitude"] = longitudes
                addresses_df["geocode_status"] = geocode_statuses

                matched_count = int((addresses_df["geocode_status"] == "matched").sum())
                print(f"{len(addresses_df)}行中{matched_count}行を変換した。")
                display(addresses_df.head())
                poc_status["csv_batch"] = matched_count > 0


In [ ]:
# ===== 検証結果サマリ =====


def _fmt(value):
    if value is True:
        return "OK"
    if value is False:
        return "NG"
    return "未実施"


LABELS = [
    ("repo_fetch", "公式repo取得"),
    ("abrdb_build", "abrdbビルド"),
    ("abrg_build", "abrgビルド"),
    ("postgres", "PostgreSQL"),
    ("okinawa_init", "沖縄県限定init"),
    ("abr_fetch", "ABRデータ取得"),
    ("postgres_import", "PostgreSQL import"),
    ("duckdb_cache", "DuckDB cache build"),
    ("geocode", "公開住所geocode"),
    ("csv_batch", "CSV一括変換"),
]

print("ABR Geocoder方式PoC 検証結果サマリ")
for key, label in LABELS:
    print(f"  {label:<20}: {_fmt(poc_status.get(key))}")

print()
_stage_order = [k for k, _ in LABELS if k != "csv_batch"]
_first_failed = next((k for k in _stage_order if poc_status.get(k) is False), None)
_label_map = dict(LABELS)

if poc_status.get("geocode"):
    print("公式ABR Geocoder方式を次の検証へ進められる材料がある。")
elif _first_failed:
    print(f"{_label_map[_first_failed]}の段階でColab上では成立しなかった。")
else:
    print("判定に必要な情報が不足している（いずれかのセルが未実行の可能性がある）。")

print()
print("この結果は本Notebook内の検証にとどまり、sheltermatch本体へは反映していません。")
print("このNotebookはJageocoderの代替採用を決定するものではなく、公式構成がColab上で成立するかの確認用です。")
